# Homework 1: Intro and Data Sources


# Question 1: [Index] S&P 500 Stocks Added to the Index


Which year had the highest number of additions (starting from 2020)?



In [36]:
import pandas as pd
import requests

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

# Getting the data in Wikipedia
response = requests.get(url, headers=headers)
tables = pd.read_html(response.text)
df = tables[0]

# Extracting the years
df['Date added'] = pd.to_datetime(df['Date added'], errors='coerce')
df['Year Added'] = df['Date added'].dt.year

# Filtering and founding the year with most additions starting in 2020
recent_additions = df[df['Year Added'] >= 2020]
additions_per_year = recent_additions['Year Added'].value_counts()
best_year = additions_per_year.idxmax()

print(f"Year with the most additions: {int(best_year)} with {additions_per_year.max()} additions.")

#Adicional
current_year = 2026
older_than_20_years = df[df['Year Added'] <= (current_year - 20)].shape[0]
print(f"Additions in S&P 500 older than 20 year: {older_than_20_years}")

Year with the most additions: 2025 with 18 additions.
Additions in S&P 500 older than 20 year: 227


/tmp/ipykernel_543/3052767636.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


## Question 2. [Macro] Indexes YTD (as of 21 August 2026)


How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?



In [37]:
import yfinance as yf
import pandas as pd

tickers = [
    '^GSPC', '000001.SS', '^HSI', '^AXJO', '^NSEI',
    '^GSPTSE', '^GDAXI', '^FTSE', '^N225', '^MXX', '^BVSP'
]

# Download data and set auto_adjust to True to avoid the warning
data = yf.download(tickers, start='2026-01-01', end='2026-08-22', auto_adjust=True)['Close']

# Forward and backward fill to handle different market holidays across countries
data = data.ffill().bfill()

# Calculate YTD return
ytd_returns = (data.iloc[-1] / data.iloc[0]) - 1

sp500_return = ytd_returns['^GSPC']
better_than_sp500 = ytd_returns[ytd_returns > sp500_return]

print(f"S&P 500 YTD Return: {sp500_return:.2%}")
print(f"Number of indexes outperforming S&P 500: {len(better_than_sp500)} out of 10")
print("\nIndexes that outperformed the S&P 500:")
print(better_than_sp500.apply(lambda x: f"{x:.2%}"))

[*********************100%***********************]  11 of 11 completed


S&P 500 YTD Return: 11.90%
Number of indexes outperforming S&P 500: 2 out of 10

Indexes that outperformed the S&P 500:
Ticker
^GSPTSE    14.86%
^N225      27.36%
dtype: object


## Question 3. [Index] S&P 500 Market Corrections Analysis


Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.



In [38]:
import yfinance as yf
import pandas as pd

# Download S&P 500 historical data
df = pd.DataFrame(yf.download('^GSPC', start='1950-01-01', auto_adjust=True)['Close'])
df.columns = ['Close']

# Identify All-Time Highs (ATH)
df['ATH'] = df['Close'].cummax()

corrections = []

# Group periods by ATH value and find drawdowns
for ath_val, group in df.groupby('ATH'):
    if len(group) > 1: # Ensure there are days below ATH
        min_close = group['Close'].min()
        drawdown_pct = (ath_val - min_close) / ath_val * 100

        # Filter for corrections of at least 5%
        if drawdown_pct >= 5.0:
            min_date = group['Close'].idxmin()
            ath_date = group.index[0]

            # Duration in calendar days from peak to trough
            duration = (min_date - ath_date).days

            corrections.append({
                'Peak Date': ath_date,
                'Trough Date': min_date,
                'Drawdown (%)': drawdown_pct,
                'Duration (Days)': duration
            })

df_corrections = pd.DataFrame(corrections)

# Calculate 25th, 50th (median) and 75th percentiles
percentiles = df_corrections[['Drawdown (%)', 'Duration (Days)']].quantile([0.25, 0.50, 0.75])
print("Correction Statistics (>5%):")
print(percentiles)

[*********************100%***********************]  1 of 1 completed

Correction Statistics (>5%):
      Drawdown (%)  Duration (Days)
0.25      6.234677            22.00
0.50      7.986358            40.50
0.75     14.019826            86.25


## Question 4. [Stocks] Earnings Surprise Analysis for Amazon (AMZN)


Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.



In [40]:
import yfinance as yf
import pandas as pd

ticker = 'AMZN'
amzn = yf.Ticker(ticker)

# Load earnings data
earnings = amzn.get_earnings_dates()

# Remove timezone to align with price data and drop null values
earnings.index = earnings.index.tz_localize(None).normalize()
earnings = earnings.dropna(subset=['Surprise(%)'])
earnings = earnings[earnings.index < pd.Timestamp.now()] # Ignore future dates

# Download complete historical price data
prices = pd.DataFrame(yf.download(ticker, start='2020-01-01', auto_adjust=True)['Close'])
prices.columns = ['Close']
prices.index = prices.index.tz_localize(None).normalize()

results = []

# Calculate 2-day returns (Day 3 / Day 1 - 1)
for date, row in earnings.iterrows():
    if date in prices.index:
        loc = prices.index.get_loc(date)
        if loc > 0 and loc < len(prices) - 1:
            p1 = prices.iloc[loc - 1]['Close'] # Day prior to announcement (Day 1)
            p3 = prices.iloc[loc + 1]['Close'] # Day after announcement (Day 3)
            ret = (p3 / p1) - 1
            results.append({
                'Date': date,
                'Surprise': row['Surprise(%)'],
                'Return': ret
            })

df_results = pd.DataFrame(results)

# Filter positive surprises and calculate median and correlation
positive_surprises = df_results[df_results['Surprise'] > 0]
median_return = positive_surprises['Return'].median()
correlation = df_results['Surprise'].corr(df_results['Return'])

print("")
print(f"Median 2-day return after positive surprise: {median_return:.2%}")
print(f"Correlation between surprise magnitude and 2-day return: {correlation:.4f}")

[*********************100%***********************]  1 of 1 completed


Median 2-day return after positive surprise: 0.35%
Correlation between surprise magnitude and 2-day return: 0.2191
